# SIGMOD Exp 2: Crossover Analysis

This notebook fixes a read-heavy maintained-state profile and sweeps three factors that matter for `SNAP` vs `IVMH` vs MVHT tradeoffs:

1. Delta-scan share
2. Update intensity
3. History-scan reuse share

The output is a 3-panel line figure over total latency.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_REPEAT,
    SIGMOD_TRIM,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    append_csv_rows,
    current_run_stamp,
    display_name,
    ensure_dirs,
    run_checked,
    snapshot_csv,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_crossover').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
BASELINES = {'naive', 'ivmh'}
TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.0001,
    'probe_ratio': 0.01,
    'txn_gc_ratio': 0.05,
    'repeat': SIGMOD_REPEAT,
    'trim': SIGMOD_TRIM,
    'readable_every': SIGMOD_READABLE_EVERY,
    'force_rerun': False,
    'timeout_sec': 900,
}

SWEEP = {
    'update_share': 0.20,
    'history_values': [0.0, 0.02, 0.04, 0.06, 0.08, 0.10],
    'delta_values': [0.00, 0.02, 0.04, 0.06, 0.08, 0.10],
}

REPEAT = CONFIG['repeat']
TXN_NUM = CONFIG['txn_count']
TIMEOUT_SEC = CONFIG['timeout_sec']
BASE_ARGS = [
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
]
PLOT_SERIES = [
    ('naive', ''),
    ('ivmh', ''),
    ('heap', 'Write Repair'),
    ('chain', 'Write Repair'),
    ('par', 'Write Repair'),
]
STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}


def token(value):
    if isinstance(value, float):
        return f'{value:g}'.replace('.', 'p')
    return str(value)


RUN_STAMP = current_run_stamp()

CONFIG_TAG = '_'.join([
    f"wc{token(CONFIG['warehouse_count'])}",
    f"tc{token(CONFIG['txn_count'])}",
    f"bn{token(CONFIG['bucket_num'])}",
    f"ur{token(CONFIG['update_ratio'])}",
    f"pr{token(CONFIG['probe_ratio'])}",
    f"gc{token(CONFIG['txn_gc_ratio'])}",
    f"re{token(CONFIG['readable_every'])}",
    f"rep{token(CONFIG['repeat'])}",
])

RUN_TAG = f'{CONFIG_TAG}_{RUN_STAMP}'

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('CONFIG :', CONFIG)
print('STAMP  :', RUN_STAMP)
print('CACHE  :', CONFIG_TAG)
print('TAG    :', RUN_TAG)


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')

In [ ]:
def merge_args(base, extra):
    merged = {}
    for args in (base, extra):
        it = iter(args)
        for token in it:
            merged[token] = next(it)
    out = []
    for k, v in merged.items():
        out.extend([k, v])
    return out


def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def collapse_repairs(df, table_type):
    if table_type in BASELINES:
        collapsed = df.groupby(['table_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'duration_ms']]
    return df


def expected_result_pairs():
    return {
        ('naive', ''),
        ('ivmh', ''),
        ('heap', 'No Repair'),
        ('heap', 'Read Repair'),
        ('heap', 'Write Repair'),
        ('chain', 'No Repair'),
        ('chain', 'Read Repair'),
        ('chain', 'Write Repair'),
        ('par', 'No Repair'),
        ('par', 'Read Repair'),
        ('par', 'Write Repair'),
    }


def normalized_sweep_value(value):
    return round(float(value), 10)


def latest_snapshot_csv(csv_stem):
    candidates = list(DATA_DIR.glob(f'{csv_stem}_{CONFIG_TAG}_*.csv'))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def load_progress_frame(csv_stem, progress_path):
    if progress_path.exists():
        print('Resuming from progress CSV:', progress_path.name)
        return pd.read_csv(progress_path, keep_default_na=False)
    seed_path = latest_snapshot_csv(csv_stem)
    if seed_path is None:
        return pd.DataFrame()
    seeded = pd.read_csv(seed_path, keep_default_na=False)
    seeded.to_csv(progress_path, index=False)
    print('Seeded progress CSV from:', seed_path.name)
    return seeded


def classify_progress_values(df, x_col):
    if df.empty or x_col not in df.columns:
        return set(), set()
    expected = expected_result_pairs()
    tmp = df.copy()
    tmp[x_col] = tmp[x_col].astype(float).map(normalized_sweep_value)
    tmp['repair_type'] = tmp['repair_type'].fillna('')
    complete = set()
    partial = set()
    for value, group in tmp.groupby(x_col):
        seen = set(zip(group['table_type'], group['repair_type']))
        if expected.issubset(seen):
            complete.add(value)
        else:
            partial.add(value)
    return complete, partial


def dedup_result_rows(df, x_col):
    if df.empty or x_col not in df.columns:
        return df
    out = df.copy()
    out['repair_type'] = out['repair_type'].fillna('')
    out[x_col] = out[x_col].astype(float).map(normalized_sweep_value)
    return out.drop_duplicates(subset=[x_col, 'table_type', 'repair_type'], keep='last').copy()


def run_single(extra_args):
    args = merge_args(BASE_ARGS, extra_args)
    rows = []
    for table_type in TABLE_TYPES:
        trials = []
        print('  table=', table_type)
        for trial in range(REPEAT):
            result = run_checked([str(BIN), *args, '--table-type', table_type], ROOT, quiet=True, timeout=TIMEOUT_SEC)
            df = parse_result(result.stdout, table_type)
            if df.empty:
                raise RuntimeError(f'No parsed rows for {table_type}')
            df['tx_type'] = df['tx_type'].replace(TX_MAP)
            total = df.groupby('repair_type', as_index=False)['duration_ms'].sum()
            total['table_type'] = table_type
            total['trial'] = trial
            trials.append(total)
        df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
        df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
        rows.append(collapse_repairs(df_avg, table_type))
    out = pd.concat(rows, ignore_index=True)
    out['total_ms'] = out['duration_ms'] / TXN_NUM
    return out


def sweep_factor(csv_stem, x_col, values, make_args_fn):
    progress_path = DATA_DIR / f'{csv_stem}_{CONFIG_TAG}_progress.csv'
    csv_path = DATA_DIR / f'{csv_stem}_{RUN_TAG}.csv'
    if CONFIG['force_rerun'] and progress_path.exists():
        progress_path.unlink()
    existing = load_progress_frame(csv_stem, progress_path)
    existing = dedup_result_rows(existing, x_col)
    if not existing.empty:
        existing.to_csv(progress_path, index=False)
    done_values, partial_values = classify_progress_values(existing, x_col)

    for value in values:
        value_key = normalized_sweep_value(value)
        if value_key in done_values:
            print(f'Skipping cached {x_col}={value}')
            continue
        if value_key in partial_values:
            print(f'Rerunning incomplete {x_col}={value}')
            existing = existing[
                existing[x_col].astype(float).map(normalized_sweep_value) != value_key
            ].copy()
            existing.to_csv(progress_path, index=False)
            done_values.discard(value_key)
            partial_values.discard(value_key)
        print(f'Running {x_col}={value}')
        df = run_single(make_args_fn(value))
        df[x_col] = value
        append_csv_rows(progress_path, df)
        existing = pd.read_csv(progress_path, keep_default_na=False)
        existing = dedup_result_rows(existing, x_col)
        existing.to_csv(progress_path, index=False)
        done_values, partial_values = classify_progress_values(existing, x_col)
        print('Appended', progress_path.name)

    out = pd.read_csv(progress_path, keep_default_na=False)
    out = dedup_result_rows(out, x_col)
    out.to_csv(progress_path, index=False)
    snapshot_csv(progress_path, csv_path)
    print('Saved', csv_path.name)
    return out


def history_args(history_ratio):
    update_share = SWEEP['update_share']
    recent_read_share = 1.0 - update_share
    return [
        '--txn-update-ratio', str(update_share),
        '--txn-probe-ratio', str(recent_read_share / 2.0),
        '--txn-scan-ratio', str(recent_read_share / 2.0),
        '--txn-delta-ratio', '0.0',
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', str(history_ratio),
        '--probe-history-ratio', str(history_ratio),
    ]


def delta_args(delta_ratio):
    update_share = SWEEP['update_share']
    remaining_reads = 1.0 - update_share - delta_ratio
    return [
        '--txn-update-ratio', str(update_share),
        '--txn-probe-ratio', str(remaining_reads / 2.0),
        '--txn-scan-ratio', str(remaining_reads / 2.0),
        '--txn-delta-ratio', str(delta_ratio),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', '0.0',
        '--probe-history-ratio', '0.0',
    ]


df_history = sweep_factor('sigmod_exp2_history', 'history_ratio', SWEEP['history_values'], history_args)
df_history['history_pct'] = df_history['history_ratio'] * 100.0
df_delta = sweep_factor('sigmod_exp2_delta', 'delta_ratio', SWEEP['delta_values'], delta_args)
df_delta['delta_pct'] = df_delta['delta_ratio'] * 100.0

display(df_history.head())


In [ ]:
def plot_one(ax, df, x_col, xlabel, title, include_snap, legend_loc, legend_bbox=None):
    for key in PLOT_SERIES:
        if not include_snap and key == ('naive', ''):
            continue
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)]
        sub = dedup_result_rows(sub, x_col).sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    if legend_bbox is None:
        ax.legend(loc=legend_loc, ncol=1, framealpha=0.95)
    else:
        ax.legend(loc=legend_loc, bbox_to_anchor=legend_bbox, ncol=1, framealpha=0.95)


def render_pair(df, x_col, xlabel, stem, base_title):
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.1), sharey=False)
    plot_one(axes[0], df, x_col, xlabel, f'{base_title} (with SNAP)', True, 'center left')
    plot_one(axes[1], df, x_col, xlabel, f'{base_title} (without SNAP)', False, 'upper left')
    fig.tight_layout()
    out_pdf = FIGS_DIR / f'{stem}_{RUN_TAG}.pdf'
    fig.savefig(out_pdf, format='pdf')
    plt.show()
    print('Saved', out_pdf)


def render_single(df, x_col, xlabel, stem, title, include_snap):
    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.1))
    legend_loc = 'center left' if include_snap else 'upper left'
    plot_one(ax, df, x_col, xlabel, title, include_snap, legend_loc)
    fig.tight_layout()
    out_pdf = FIGS_DIR / f'{stem}_{RUN_TAG}.pdf'
    fig.savefig(out_pdf, format='pdf')
    plt.show()
    print('Saved', out_pdf)


render_pair(df_history, 'history_pct', 'Historical Read Percentage (%)', 'sigmod_exp2_history_crossover', 'Recent-to-Historical Sweep')
render_pair(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'sigmod_exp2_delta_crossover', 'Recent-to-Delta Sweep')


In [ ]:
render_single(df_history, 'history_pct', 'Historical Read Percentage (%)', 'sigmod_exp2_history_with_snap', 'Recent-to-Historical Sweep (with SNAP)', True)
render_single(df_history, 'history_pct', 'Historical Read Percentage (%)', 'sigmod_exp2_history_without_snap', 'Recent-to-Historical Sweep (without SNAP)', False)


In [ ]:
render_single(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'sigmod_exp2_delta_with_snap', 'Recent-to-Delta Sweep (with SNAP)', True)
render_single(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'sigmod_exp2_delta_without_snap', 'Recent-to-Delta Sweep (without SNAP)', False)
